# ✂️ AutoClips — sin instalar nada

Saca los **momentos graciosos y épicos** de un directo (o un directo resubido de Twitch) y te
los da listos para TikTok. Funciona desde el **navegador del móvil o del PC**, gratis, en los
ordenadores de Google.

### Cómo se usa
1. Pulsa **▶** en el paso **1** y espera a que salga ✅ (1–2 minutos).
2. Pulsa **▶** en el paso **2** y toca el botón **Abrir AutoClips**.
3. Pega el enlace, elige cuántos clips quieres y pulsa **Sacar clips**. Descárgalos al terminar.

💡 *Opcional, más rápido:* menú **Entorno de ejecución → Cambiar tipo de entorno → GPU T4**
antes del paso 1 (la IA que detecta risas va mucho más rápida).

⚠️ Deja esta pestaña abierta mientras trabaja. Si Google desconecta la sesión, vuelve a pulsar ▶ en 1 y 2.

In [ ]:
#@title 1️⃣ Preparar AutoClips (pulsa ▶ y espera a que salga ✅)
import os, subprocess, sys
CODIGO = {
"app.py": "\"\"\"AutoClips — servidor web local.\n\nUso:\n    python app.py                 # abre http://127.0.0.1:5000\n    python app.py --red           # accesible desde el móvil en la misma wifi\n\nSi existe la variable de entorno AUTOCLIPS_CLAVE, la web pide esa contraseña\n(útil al publicarla en internet, p. ej. en Hugging Face Spaces).\n\"\"\"\n\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport hmac\nimport os\nimport re\nimport shutil\nimport threading\nimport time\nimport traceback\nimport uuid\nimport webbrowser\n\nfrom flask import Flask, abort, jsonify, redirect, render_template, request, send_from_directory\nfrom werkzeug.utils import secure_filename\n\nfrom clipper import ESTILOS, FORMATOS, ClipError, Opciones, ia_voz_disponible, procesar\n\nRAIZ = os.path.dirname(os.path.abspath(__file__))\nTRABAJOS = os.path.join(RAIZ, \"trabajos\")\nHORAS_CADUCIDAD = 24\nCLAVE = os.environ.get(\"AUTOCLIPS_CLAVE\", \"\")\nCOOKIE = \"autoclips_sesion\"\n\napp = Flask(__name__)\napp.config[\"MAX_CONTENT_LENGTH\"] = 8 * 1024 ** 3  # subidas de hasta 8 GB\n\ntrabajos: dict[str, dict] = {}\ncerrojo = threading.Lock()\n# Un trabajo pesado a la vez para no saturar el ordenador.\nturno = threading.Semaphore(1)\n\n\ndef _ficha() -> str:\n    return hmac.new(CLAVE.encode(), b\"autoclips\", hashlib.sha256).hexdigest()\n\n\n@app.before_request\ndef _proteger():\n    if not CLAVE or request.endpoint == \"entrar\":\n        return None\n    if hmac.compare_digest(request.cookies.get(COOKIE, \"\"), _ficha()):\n        return None\n    if request.path.startswith(\"/api/\"):\n        return jsonify(error=\"Sesión caducada: recarga la página y escribe la contraseña.\"), 401\n    return render_template(\"entrar.html\", error=False), 401\n\n\n@app.post(\"/entrar\")\ndef entrar():\n    if not CLAVE or not hmac.compare_digest(request.form.get(\"clave\", \"\"), CLAVE):\n        return render_template(\"entrar.html\", error=True), 401\n    respuesta = redirect(\"./\")\n    seguro = request.is_secure or request.headers.get(\"X-Forwarded-Proto\") == \"https\"\n    respuesta.set_cookie(COOKIE, _ficha(), max_age=30 * 24 * 3600, httponly=True,\n                         secure=seguro, samesite=\"None\" if seguro else \"Lax\")\n    return respuesta\n\n\ndef _actualizar(id_: str, **datos) -> None:\n    with cerrojo:\n        trabajos[id_].update(datos)\n\n\ndef _procesar(id_: str, origen: str, opciones: Opciones) -> None:\n    carpeta = os.path.join(TRABAJOS, id_)\n    _actualizar(id_, estado=\"en_cola\", mensaje=\"Esperando turno…\")\n    with turno:\n        try:\n            _actualizar(id_, estado=\"procesando\")\n            resultado = procesar(\n                origen, carpeta, opciones,\n                lambda p, m: _actualizar(id_, progreso=round(p, 1), mensaje=m),\n            )\n            shutil.rmtree(os.path.join(carpeta, \"fuente\"), ignore_errors=True)\n            if origen.startswith(carpeta):\n                os.remove(origen)\n            _actualizar(id_, estado=\"listo\", progreso=100, mensaje=\"¡Clips listos!\",\n                        resultado=resultado.a_dict(), titulo=resultado.titulo)\n        except ClipError as exc:\n            _actualizar(id_, estado=\"error\", mensaje=str(exc))\n        except Exception as exc:  # noqa: BLE001\n            traceback.print_exc()\n            _actualizar(id_, estado=\"error\", mensaje=f\"Error inesperado: {exc}\")\n\n\ndef _limpiar_antiguos() -> None:\n    if not os.path.isdir(TRABAJOS):\n        return\n    limite = time.time() - HORAS_CADUCIDAD * 3600\n    for nombre in os.listdir(TRABAJOS):\n        ruta = os.path.join(TRABAJOS, nombre)\n        if os.path.isdir(ruta) and os.path.getmtime(ruta) < limite:\n            shutil.rmtree(ruta, ignore_errors=True)\n\n\ndef _numero(valor, defecto, minimo, maximo, tipo=float):\n    try:\n        return max(minimo, min(maximo, tipo(valor)))\n    except (TypeError, ValueError):\n        return defecto\n\n\n@app.get(\"/\")\ndef inicio():\n    return render_template(\"index.html\", formatos=FORMATOS, estilos=ESTILOS, ia_voz=ia_voz_disponible())\n\n\n@app.post(\"/api/trabajos\")\ndef crear_trabajo():\n    f = request.form\n    url = (f.get(\"url\") or \"\").strip()\n    subida = request.files.get(\"archivo\")\n    if not url and not (subida and subida.filename):\n        return jsonify(error=\"Pega un enlace o sube un vídeo.\"), 400\n    if url and not re.match(r\"^https?://\", url):\n        return jsonify(error=\"El enlace debe empezar por http:// o https://\"), 400\n\n    id_ = uuid.uuid4().hex[:12]\n    carpeta = os.path.join(TRABAJOS, id_)\n    os.makedirs(carpeta, exist_ok=True)\n\n    duracion = f.get(\"duracion\", \"auto\")\n    opciones = Opciones(\n        cantidad=_numero(f.get(\"cantidad\"), 5, 1, 50, int),\n        duracion=\"auto\" if duracion == \"auto\" else _numero(duracion, 30, 5, 600),\n        formato=f.get(\"formato\") if f.get(\"formato\") in FORMATOS else \"vertical_fondo\",\n        estilo=f.get(\"estilo\") if f.get(\"estilo\") in ESTILOS else \"todo\",\n        ia_voz=f.get(\"ia_voz\") == \"on\",\n        subtitulos=f.get(\"subtitulos\") == \"on\",\n        escenas=f.get(\"escenas\", \"on\") == \"on\",\n        minutos_directo=_numero(f.get(\"minutos_directo\"), 10, 1, 240),\n    )\n    origen = url\n    titulo = url\n    if subida and subida.filename:\n        nombre = secure_filename(subida.filename) or \"video.mp4\"\n        origen = os.path.join(carpeta, \"subida_\" + nombre)\n        subida.save(origen)\n        titulo = opciones.titulo = subida.filename\n\n    with cerrojo:\n        trabajos[id_] = {\"id\": id_, \"estado\": \"en_cola\", \"progreso\": 0,\n                         \"mensaje\": \"En cola…\", \"titulo\": titulo, \"resultado\": None}\n    threading.Thread(target=_procesar, args=(id_, origen, opciones), daemon=True).start()\n    return jsonify(id=id_)\n\n\n@app.get(\"/api/trabajos/<id_>\")\ndef ver_trabajo(id_: str):\n    with cerrojo:\n        t = trabajos.get(id_)\n        if not t:\n            abort(404)\n        return jsonify(t)\n\n\n@app.get(\"/trabajos/<id_>/<path:archivo>\")\ndef archivo_trabajo(id_: str, archivo: str):\n    if not re.fullmatch(r\"[0-9a-f]{12}\", id_) or not re.fullmatch(r\"[\\w.-]+\", archivo):\n        abort(404)\n    descarga = request.args.get(\"descargar\") == \"1\"\n    return send_from_directory(os.path.join(TRABAJOS, id_), archivo, as_attachment=descarga)\n\n\ndef main() -> None:\n    p = argparse.ArgumentParser(description=\"AutoClips: genera clips automáticamente.\")\n    p.add_argument(\"--puerto\", type=int, default=5000)\n    p.add_argument(\"--red\", action=\"store_true\",\n                   help=\"Permitir acceso desde otros dispositivos de tu red (móvil).\")\n    p.add_argument(\"--sin-navegador\", action=\"store_true\", help=\"No abrir el navegador.\")\n    a = p.parse_args()\n\n    os.makedirs(TRABAJOS, exist_ok=True)\n    _limpiar_antiguos()\n    host = \"0.0.0.0\" if a.red else \"127.0.0.1\"\n    direccion = f\"http://127.0.0.1:{a.puerto}\"\n    print(f\"\\n  AutoClips funcionando en {direccion}\\n  (Ctrl+C para cerrar)\\n\")\n    if not a.sin_navegador:\n        threading.Timer(1.2, lambda: webbrowser.open(direccion)).start()\n    app.run(host=host, port=a.puerto, threaded=True)\n\n\nif __name__ == \"__main__\":\n    main()\n",
"clips.py": "\"\"\"AutoClips desde la terminal.\n\nEjemplos:\n    python clips.py \"https://www.twitch.tv/videos/123456789\" -n 10 --estilo graciosos\n    python clips.py \"https://www.youtube.com/watch?v=XXXX\" -n 5 -d 30\n    python clips.py \"https://www.twitch.tv/canal\" --directo 15 -n 3\n    python clips.py mi_video.mp4 -n 8 --subtitulos\n\"\"\"\n\nfrom __future__ import annotations\n\nimport argparse\nimport os\nimport sys\nimport time\n\nfrom clipper import ESTILOS, FORMATOS, ClipError, Opciones, procesar\n\n\ndef barra(pct: float, msg: str) -> None:\n    llenos = int(pct / 4)\n    sys.stdout.write(f\"\\r[{'#' * llenos}{'.' * (25 - llenos)}] {pct:5.1f}%  {msg[:50]:<50}\")\n    sys.stdout.flush()\n\n\ndef main() -> int:\n    p = argparse.ArgumentParser(description=\"Saca clips de los mejores momentos de un vídeo o directo.\")\n    p.add_argument(\"fuente\", help=\"Enlace (Twitch, YouTube, Kick, TikTok…) o archivo de vídeo\")\n    p.add_argument(\"-n\", \"--cantidad\", type=int, default=5, help=\"Número de clips (por defecto 5)\")\n    p.add_argument(\"-d\", \"--duracion\", default=\"auto\",\n                   help=\"Segundos por clip o 'auto' (15-60 s según el momento; por defecto)\")\n    p.add_argument(\"-f\", \"--formato\", choices=list(FORMATOS), default=\"vertical_fondo\")\n    p.add_argument(\"-e\", \"--estilo\", choices=list(ESTILOS), default=\"todo\",\n                   help=\"Qué momentos priorizar: todo, graciosos o epicos\")\n    p.add_argument(\"-o\", \"--salida\", default=None, help=\"Carpeta de salida\")\n    p.add_argument(\"--subtitulos\", action=\"store_true\", help=\"Subtítulos automáticos\")\n    p.add_argument(\"--sin-ia\", action=\"store_true\", help=\"No usar la IA de voz para detectar risas\")\n    p.add_argument(\"--sin-chat\", action=\"store_true\", help=\"No leer el chat del stream\")\n    p.add_argument(\"--rapido\", action=\"store_true\", help=\"No analizar cortes de cámara\")\n    p.add_argument(\"--directo\", type=float, default=10, metavar=\"MIN\",\n                   help=\"Minutos a grabar si el enlace es un directo en emisión (por defecto 10)\")\n    a = p.parse_args()\n\n    op = Opciones(\n        cantidad=a.cantidad,\n        duracion=\"auto\" if a.duracion == \"auto\" else float(a.duracion),\n        formato=a.formato, estilo=a.estilo, subtitulos=a.subtitulos,\n        ia_voz=not a.sin_ia, chat=not a.sin_chat, escenas=not a.rapido,\n        minutos_directo=a.directo,\n    )\n    salida = a.salida or os.path.join(\"clips\", time.strftime(\"%Y%m%d_%H%M%S\"))\n    t0 = time.time()\n    try:\n        r = procesar(a.fuente, salida, op, barra)\n    except ClipError as exc:\n        print(f\"\\nError: {exc}\")\n        return 1\n    print(f\"\\n\\n{len(r.clips)} clips de «{r.titulo}» en {time.time() - t0:.0f} s\")\n    print(f\"Analizado con: {', '.join(r.senales)}\")\n    for aviso in r.avisos:\n        print(f\"Aviso: {aviso}\")\n    for c in r.clips:\n        print(f\"  #{c.numero} {c.archivo}  {c.inicio:8.1f}s → {c.fin:8.1f}s  \"\n              f\"({c.fin - c.inicio:4.0f}s)  {c.puntuacion}%  {c.motivo}\")\n    print(f\"Carpeta: {os.path.abspath(salida)}  (todo junto en clips.zip)\")\n    return 0\n\n\nif __name__ == \"__main__\":\n    sys.exit(main())\n",
"templates/index.html": "<!doctype html>\n<html lang=\"es\">\n<head>\n<meta charset=\"utf-8\">\n<meta name=\"viewport\" content=\"width=device-width, initial-scale=1\">\n<title>AutoClips</title>\n<style>\n  :root {\n    --fondo: #0f1115; --panel: #181b22; --borde: #2a2f3a; --texto: #e8eaf0;\n    --suave: #9aa3b2; --acento: #ff4d6d; --acento2: #ff8a3d; --ok: #2ecc71;\n  }\n  * { box-sizing: border-box; }\n  body { margin: 0; background: var(--fondo); color: var(--texto);\n         font: 16px/1.5 system-ui, -apple-system, \"Segoe UI\", Roboto, sans-serif; }\n  main { max-width: 1100px; margin: 0 auto; padding: 24px 16px 64px; }\n  h1 { margin: 0 0 4px; font-size: 2rem; }\n  h1 span { background: linear-gradient(90deg, var(--acento), var(--acento2));\n            -webkit-background-clip: text; background-clip: text; color: transparent; }\n  .sub { color: var(--suave); margin: 0 0 24px; }\n  .panel { background: var(--panel); border: 1px solid var(--borde); border-radius: 14px; padding: 20px; }\n  label { display: block; font-weight: 600; margin-bottom: 6px; }\n  input[type=url], input[type=number], select {\n    width: 100%; padding: 12px; border-radius: 10px; border: 1px solid var(--borde);\n    background: #0d0f13; color: var(--texto); font-size: 1rem; }\n  .fila { display: grid; gap: 16px; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr)); margin-top: 16px; }\n  .ayuda { color: var(--suave); font-size: .85rem; margin-top: 4px; }\n  .check { display: flex; gap: 8px; align-items: center; font-weight: 400; margin: 0; }\n  .o { text-align: center; color: var(--suave); margin: 10px 0; font-size: .9rem; }\n  input[type=file] { color: var(--suave); width: 100%; }\n  .chips { display: flex; flex-wrap: wrap; gap: 6px; margin-top: 8px; }\n  .chip { border: 1px solid var(--borde); background: transparent; color: var(--texto);\n          border-radius: 999px; padding: 4px 12px; cursor: pointer; font-size: .9rem; }\n  .chip.activo { border-color: var(--acento); background: rgba(255,77,109,.15); }\n  button.principal { margin-top: 20px; width: 100%; padding: 14px; font-size: 1.1rem; font-weight: 700;\n         border: 0; border-radius: 12px; color: #fff; cursor: pointer;\n         background: linear-gradient(90deg, var(--acento), var(--acento2)); }\n  button.principal:disabled { opacity: .5; cursor: wait; }\n  #estado { margin-top: 20px; display: none; }\n  .barra { height: 12px; background: #0d0f13; border-radius: 999px; overflow: hidden; border: 1px solid var(--borde); }\n  .barra > div { height: 100%; width: 0; transition: width .4s;\n                 background: linear-gradient(90deg, var(--acento), var(--acento2)); }\n  #mensaje { margin-top: 8px; color: var(--suave); white-space: pre-wrap; }\n  .error { color: #ff6b6b !important; }\n  #resultado { margin-top: 24px; display: none; }\n  .cabecera { display: flex; flex-wrap: wrap; justify-content: space-between; align-items: center; gap: 12px; }\n  .boton { display: inline-block; padding: 10px 16px; border-radius: 10px; text-decoration: none;\n           font-weight: 600; color: #fff; background: var(--ok); }\n  .boton.sec { background: #2a2f3a; }\n  .rejilla { display: grid; gap: 16px; margin-top: 16px; grid-template-columns: repeat(auto-fill, minmax(240px, 1fr)); }\n  .clip { background: var(--panel); border: 1px solid var(--borde); border-radius: 14px; overflow: hidden; }\n  .clip video { width: 100%; display: block; background: #000; max-height: 420px; }\n  .clip .info { padding: 12px; }\n  .clip .meta { color: var(--suave); font-size: .85rem; }\n  .clip .motivo { font-size: .85rem; margin-top: 6px; }\n  .clip .texto { font-size: .85rem; margin: 6px 0; max-height: 4.5em; overflow: hidden; }\n  .nota { float: right; font-weight: 700; color: var(--acento2); }\n  .avisos { color: #f5c26b; font-size: .9rem; }\n</style>\n</head>\n<body>\n<main>\n  <h1>✂️ <span>AutoClips</span></h1>\n  <p class=\"sub\">Pega el enlace de un directo (o resubido), elige cuántos clips quieres y te saca los momentos graciosos y épicos listos para TikTok. 100% gratis.</p>\n\n  <form id=\"form\" class=\"panel\">\n    <label for=\"url\">Enlace del vídeo o del directo</label>\n    <input type=\"url\" id=\"url\" name=\"url\" placeholder=\"https://www.twitch.tv/videos/…\">\n    <div class=\"ayuda\">Directos resubidos de Twitch (twitch.tv/videos/…), YouTube, Kick, TikTok, X… y más de 1000 webs.\n      En Twitch y YouTube también lee el <b>chat</b> para saber dónde se rió o flipó la gente.</div>\n    <div class=\"o\">— o —</div>\n    <label for=\"archivo\">Sube un vídeo de tu móvil u ordenador</label>\n    <input type=\"file\" id=\"archivo\" name=\"archivo\" accept=\"video/*\">\n\n    <div class=\"fila\">\n      <div>\n        <label for=\"cantidad\">Número de clips</label>\n        <input type=\"number\" id=\"cantidad\" name=\"cantidad\" min=\"1\" max=\"50\" value=\"5\">\n        <div class=\"chips\" data-para=\"cantidad\">\n          <button type=\"button\" class=\"chip\" data-v=\"3\">3</button>\n          <button type=\"button\" class=\"chip activo\" data-v=\"5\">5</button>\n          <button type=\"button\" class=\"chip\" data-v=\"10\">10</button>\n          <button type=\"button\" class=\"chip\" data-v=\"20\">20</button>\n        </div>\n      </div>\n      <div>\n        <label for=\"duracion\">Duración de cada clip</label>\n        <select id=\"duracion\" name=\"duracion\">\n          <option value=\"auto\" selected>Automática (15–60 s)</option>\n          <option value=\"15\">15 s</option>\n          <option value=\"20\">20 s</option>\n          <option value=\"30\">30 s</option>\n          <option value=\"45\">45 s</option>\n          <option value=\"60\">1 min</option>\n          <option value=\"90\">1:30</option>\n          <option value=\"120\">2 min</option>\n        </select>\n        <div class=\"ayuda\">En automático cada clip empieza un poco antes del momento y acaba tras la reacción.</div>\n      </div>\n      <div>\n        <label for=\"estilo\">¿Qué momentos buscas?</label>\n        <select id=\"estilo\" name=\"estilo\">\n          {% for clave, nombre in estilos.items() %}<option value=\"{{ clave }}\">{{ nombre }}</option>{% endfor %}\n        </select>\n      </div>\n      <div>\n        <label for=\"formato\">Formato</label>\n        <select id=\"formato\" name=\"formato\">\n          {% for clave, nombre in formatos.items() %}<option value=\"{{ clave }}\">{{ nombre }}</option>{% endfor %}\n        </select>\n      </div>\n    </div>\n\n    <div class=\"fila\">\n      <div>\n        <label for=\"minutos_directo\">Si es un directo: minutos a grabar</label>\n        <input type=\"number\" id=\"minutos_directo\" name=\"minutos_directo\" min=\"1\" max=\"240\" value=\"10\">\n        <div class=\"ayuda\">Graba desde ahora y luego saca los clips de esa grabación.</div>\n      </div>\n      <div>\n        <label class=\"check\"><input type=\"checkbox\" name=\"ia_voz\" {% if ia_voz %}checked{% else %}disabled{% endif %}> Detectar risas con IA de voz</label>\n        <div class=\"ayuda\">{% if ia_voz %}Escucha los mejores candidatos y sube los que tienen risas o reacciones (tarda algo más).{% else %}Instala <code>faster-whisper</code> para activarlo (ver README).{% endif %}</div>\n        <label class=\"check\" style=\"margin-top:10px\"><input type=\"checkbox\" name=\"subtitulos\" {% if not ia_voz %}disabled{% endif %}> Subtítulos automáticos</label>\n        <label class=\"check\" style=\"margin-top:10px\"><input type=\"checkbox\" name=\"escenas\" checked> Analizar cortes de cámara</label>\n        <div class=\"ayuda\">Solo con vídeos subidos o directos grabados.</div>\n      </div>\n    </div>\n\n    <button class=\"principal\" id=\"enviar\" type=\"submit\">🎬 Sacar clips</button>\n  </form>\n\n  <section id=\"estado\" class=\"panel\">\n    <strong id=\"titulo\"></strong>\n    <div class=\"barra\" style=\"margin-top:10px\"><div id=\"relleno\"></div></div>\n    <div id=\"mensaje\"></div>\n  </section>\n\n  <section id=\"resultado\">\n    <div class=\"cabecera\">\n      <div>\n        <h2 style=\"margin:0\">Tus clips (del mejor al peor)</h2>\n        <div class=\"ayuda\" id=\"senales\"></div>\n        <div class=\"avisos\" id=\"avisos\"></div>\n      </div>\n      <a class=\"boton\" id=\"zip\" href=\"#\">⬇️ Descargar todos (ZIP)</a>\n    </div>\n    <div class=\"rejilla\" id=\"rejilla\"></div>\n  </section>\n</main>\n\n<script>\nconst $ = (s) => document.querySelector(s);\nconst form = $(\"#form\");\n\ndocument.querySelectorAll(\".chips\").forEach((grupo) => {\n  const campo = document.getElementById(grupo.dataset.para);\n  grupo.addEventListener(\"click\", (e) => {\n    const b = e.target.closest(\".chip\");\n    if (!b) return;\n    campo.value = b.dataset.v;\n    grupo.querySelectorAll(\".chip\").forEach((c) => c.classList.toggle(\"activo\", c === b));\n  });\n  campo.addEventListener(\"input\", () =>\n    grupo.querySelectorAll(\".chip\").forEach((c) => c.classList.toggle(\"activo\", c.dataset.v === campo.value)));\n});\n\nconst hms = (s) => {\n  s = Math.round(s);\n  const h = Math.floor(s / 3600), m = Math.floor((s % 3600) / 60), x = String(s % 60).padStart(2, \"0\");\n  return h ? `${h}:${String(m).padStart(2, \"0\")}:${x}` : `${m}:${x}`;\n};\n\nform.addEventListener(\"submit\", async (e) => {\n  e.preventDefault();\n  const datos = new FormData(form);\n  if (!form.escenas.checked) datos.set(\"escenas\", \"off\");\n  if (!datos.get(\"url\") && !$(\"#archivo\").files.length) {\n    alert(\"Pega un enlace o sube un vídeo.\");\n    return;\n  }\n  $(\"#enviar\").disabled = true;\n  $(\"#resultado\").style.display = \"none\";\n  $(\"#estado\").style.display = \"block\";\n  $(\"#mensaje\").className = \"\";\n  $(\"#mensaje\").textContent = $(\"#archivo\").files.length ? \"Subiendo vídeo…\" : \"Empezando…\";\n  $(\"#relleno\").style.width = \"0\";\n  $(\"#titulo\").textContent = datos.get(\"url\") || $(\"#archivo\").files[0].name;\n\n  try {\n    const r = await fetch(\"api/trabajos\", { method: \"POST\", body: datos });\n    const j = await r.json();\n    if (!r.ok) throw new Error(j.error || \"Error\");\n    seguir(j.id);\n  } catch (err) {\n    fallo(err.message);\n  }\n});\n\nfunction fallo(msg) {\n  $(\"#mensaje\").className = \"error\";\n  $(\"#mensaje\").textContent = \"❌ \" + msg;\n  $(\"#enviar\").disabled = false;\n}\n\nasync function seguir(id) {\n  try {\n    const r = await fetch(`api/trabajos/${id}`);\n    if (!r.ok) throw new Error(\"Se perdió el trabajo (¿se reinició el programa?)\");\n    const t = await r.json();\n    if (t.titulo) $(\"#titulo\").textContent = t.titulo;\n    $(\"#relleno\").style.width = (t.progreso || 0) + \"%\";\n    $(\"#mensaje\").textContent = `${Math.round(t.progreso || 0)}% · ${t.mensaje || \"\"}`;\n    if (t.estado === \"error\") return fallo(t.mensaje);\n    if (t.estado === \"listo\") return mostrar(id, t.resultado);\n  } catch (err) {\n    return fallo(err.message);\n  }\n  setTimeout(() => seguir(id), 1000);\n}\n\nfunction mostrar(id, res) {\n  $(\"#enviar\").disabled = false;\n  const base = `trabajos/${id}/`;\n  $(\"#zip\").href = base + res.zip + \"?descargar=1\";\n  $(\"#senales\").textContent = `${res.clips.length} clips · vídeo de ${hms(res.duracion_fuente)} · analizado por: ${res.senales.join(\", \")}`;\n  $(\"#avisos\").textContent = (res.avisos || []).join(\" \");\n  const rejilla = $(\"#rejilla\");\n  rejilla.innerHTML = \"\";\n  for (const c of res.clips) {\n    const el = document.createElement(\"div\");\n    el.className = \"clip\";\n    el.innerHTML = `\n      <video controls preload=\"none\" playsinline poster=\"${base + c.miniatura}\" src=\"${base + c.archivo}\"></video>\n      <div class=\"info\">\n        <span class=\"nota\">🔥 ${c.puntuacion}%</span>\n        <strong>#${c.numero}</strong>\n        <div class=\"meta\">${hms(c.inicio)} → ${hms(c.fin)} (${Math.round(c.fin - c.inicio)} s)</div>\n        <div class=\"motivo\"></div>\n        ${c.texto ? `<div class=\"texto\"></div>` : \"\"}\n        <a class=\"boton sec\" style=\"margin-top:8px\" href=\"${base + c.archivo}?descargar=1\">⬇️ Descargar</a>\n      </div>`;\n    el.querySelector(\".motivo\").textContent = c.motivo || \"\";\n    if (c.texto) el.querySelector(\".texto\").textContent = \"“\" + c.texto + \"”\";\n    rejilla.appendChild(el);\n  }\n  $(\"#resultado\").style.display = \"block\";\n  $(\"#resultado\").scrollIntoView({ behavior: \"smooth\" });\n}\n</script>\n</body>\n</html>\n",
"clipper/__init__.py": "from .core import ESTILOS, FORMATOS, ClipError, Opciones, Resultado, procesar\nfrom .voz import disponible as ia_voz_disponible\n\n__all__ = [\"ClipError\", \"ESTILOS\", \"FORMATOS\", \"Opciones\", \"Resultado\", \"ia_voz_disponible\", \"procesar\"]\n",
"clipper/analisis.py": "\"\"\"Busca los momentos destacados: puntúa cada medio segundo y elige los mejores.\"\"\"\n\nfrom __future__ import annotations\n\nimport re\nimport subprocess\nfrom dataclasses import dataclass, field\nfrom typing import Optional\n\nimport numpy as np\n\nfrom .chat import RETRASO_CHAT, Chat, clasificar\nfrom .medios import Progreso, ffmpeg_bin, sin_progreso\n\nPASO = 0.5  # segundos por muestra\n\nESTILOS = {\n    \"todo\": \"Todo lo destacado\",\n    \"graciosos\": \"Graciosos (risas)\",\n    \"epicos\": \"Épicos (hype, gritos, jugadas)\",\n}\n\n# Peso de cada señal según lo que busca el usuario.\nPESOS = {\n    \"todo\":      {\"voz\": 1.0, \"chat\": 1.0, \"risa_chat\": 1.1, \"hype_chat\": 1.0, \"escenas\": 0.35},\n    \"graciosos\": {\"voz\": 0.7, \"chat\": 0.7, \"risa_chat\": 1.8, \"hype_chat\": 0.4, \"escenas\": 0.2},\n    \"epicos\":    {\"voz\": 1.3, \"chat\": 1.0, \"risa_chat\": 0.5, \"hype_chat\": 1.6, \"escenas\": 0.5},\n}\n\n\n# ---------------------------------------------------------------------------\n# Señales\n# ---------------------------------------------------------------------------\n\ndef energia_audio(ruta: str, duracion: float, progreso: Progreso = sin_progreso) -> np.ndarray:\n    \"\"\"Volumen (dB) cada PASO segundos, leído en streaming para no gastar RAM.\"\"\"\n    sr = 8000\n    ventana = int(sr * PASO)\n    cmd = [ffmpeg_bin(), \"-hide_banner\", \"-nostdin\", \"-i\", ruta, \"-vn\", \"-ac\", \"1\",\n           \"-ar\", str(sr), \"-f\", \"s16le\", \"pipe:1\"]\n    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL)\n    assert proc.stdout is not None\n    resto = b\"\"\n    niveles: list[np.ndarray] = []\n    leidos = 0\n    while True:\n        datos = proc.stdout.read(ventana * 2 * 240)\n        if not datos:\n            break\n        datos = resto + datos\n        util = len(datos) - len(datos) % (ventana * 2)\n        resto = datos[util:]\n        if util:\n            x = np.frombuffer(datos[:util], dtype=np.int16).astype(np.float32) / 32768.0\n            x = x.reshape(-1, ventana)\n            niveles.append(20 * np.log10(np.sqrt(np.mean(x * x, axis=1) + 1e-10)))\n            leidos += x.shape[0]\n            if duracion > 0:\n                progreso(min(99.0, leidos * PASO * 100 / duracion), \"Escuchando el audio…\")\n    proc.wait()\n    if not niveles:\n        return np.full(max(1, int(duracion / PASO)), -60.0, dtype=np.float32)\n    return np.concatenate(niveles)\n\n\ndef cambios_de_escena(ruta: str, n: int, duracion: float,\n                      progreso: Progreso = sin_progreso) -> np.ndarray:\n    \"\"\"Cortes de cámara por muestra (vídeo reducido para ir rápido).\"\"\"\n    cmd = [ffmpeg_bin(), \"-hide_banner\", \"-nostdin\", \"-i\", ruta, \"-an\", \"-sn\",\n           \"-vf\", \"fps=3,scale=96:-2,select='gte(scene,0)',\"\n                  \"metadata=mode=print:key=lavfi.scene_score:file=-\",\n           \"-nostats\", \"-f\", \"null\", \"-\"]\n    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True,\n                            errors=\"replace\")\n    assert proc.stdout is not None\n    cortes = np.zeros(n, dtype=np.float32)\n    instante = aviso = 0.0\n    for linea in proc.stdout:\n        m = re.search(r\"pts_time:([\\d.]+)\", linea)\n        if m:\n            instante = float(m.group(1))\n            if duracion > 0 and instante - aviso >= 5:\n                aviso = instante\n                progreso(min(99.0, instante * 100 / duracion), \"Detectando cortes de cámara…\")\n        elif linea.startswith(\"lavfi.scene_score=\"):\n            try:\n                valor = float(linea.split(\"=\", 1)[1])\n            except ValueError:\n                continue\n            i = int(instante / PASO)\n            if valor > 0.3 and 0 <= i < n:\n                cortes[i] += 1\n    proc.wait()\n    return cortes\n\n\ndef suavizar(x: np.ndarray, segundos: float) -> np.ndarray:\n    k = max(1, min(len(x), int(segundos / PASO)))\n    if k <= 1:\n        return x.astype(np.float32)\n    return np.convolve(x, np.ones(k, dtype=np.float32) / k, mode=\"same\")\n\n\ndef z(x: np.ndarray) -> np.ndarray:\n    \"\"\"Normalización robusta (mediana / MAD) recortada a [-3, 4].\"\"\"\n    med = np.median(x)\n    mad = np.median(np.abs(x - med)) * 1.4826\n    if mad < 1e-6:\n        mad = float(np.std(x)) or 1.0\n    return np.clip((x - med) / mad, -3, 4)\n\n\ndef _mapa_de_calor(info: Optional[dict], n: int) -> Optional[np.ndarray]:\n    calor = (info or {}).get(\"heatmap\")\n    if not calor:\n        return None\n    curva = np.zeros(n, dtype=np.float32)\n    for tramo in calor:\n        a = int(float(tramo.get(\"start_time\", 0)) / PASO)\n        b = int(float(tramo.get(\"end_time\", 0)) / PASO)\n        curva[max(0, a):min(n, max(a + 1, b))] = float(tramo.get(\"value\", 0))\n    return curva if curva.any() else None\n\n\ndef series_chat(chat: Chat, n: int) -> dict[str, np.ndarray]:\n    \"\"\"Mensajes por muestra, adelantados RETRASO_CHAT para alinearlos con lo que pasó.\"\"\"\n    total = np.zeros(n, dtype=np.float32)\n    risa = np.zeros(n, dtype=np.float32)\n    hype = np.zeros(n, dtype=np.float32)\n    for t, texto in chat.mensajes:\n        i = int((t - RETRASO_CHAT) / PASO)\n        if 0 <= i < n:\n            total[i] += 1\n            es_risa, es_hype = clasificar(texto)\n            risa[i] += es_risa\n            hype[i] += es_hype\n    return {\"total\": total, \"risa\": risa, \"hype\": hype}\n\n\n@dataclass\nclass Analisis:\n    puntos: np.ndarray\n    volumen: np.ndarray\n    componentes: dict[str, np.ndarray] = field(default_factory=dict)\n    senales: list[str] = field(default_factory=list)\n\n\ndef puntuar(volumen: np.ndarray, info: Optional[dict] = None, chat: Optional[Chat] = None,\n            cortes: Optional[np.ndarray] = None, estilo: str = \"todo\") -> Analisis:\n    \"\"\"Combina todas las señales en una puntuación de interés por muestra.\"\"\"\n    n = len(volumen)\n    pesos = PESOS.get(estilo, PESOS[\"todo\"])\n    comp: dict[str, np.ndarray] = {}\n    senales = [\"volumen y gritos\"]\n\n    vol = suavizar(volumen, 2.0)\n    subidon = vol - suavizar(volumen, 45.0)  # gritos/risas respecto a lo que había antes\n    comp[\"voz\"] = pesos[\"voz\"] * (0.45 * z(vol) + 0.8 * z(subidon))\n\n    if chat:\n        s = series_chat(chat, n)\n        base = suavizar(s[\"total\"], 300.0) + 0.05\n        ritmo = np.log1p(suavizar(s[\"total\"], 8.0) / base)  # actividad relativa al momento del stream\n        comp[\"chat\"] = pesos[\"chat\"] * 1.3 * z(ritmo)\n        if s[\"risa\"].sum() >= 5:\n            comp[\"risa_chat\"] = pesos[\"risa_chat\"] * 1.4 * z(suavizar(s[\"risa\"], 8.0) / np.sqrt(base))\n        if s[\"hype\"].sum() >= 5:\n            comp[\"hype_chat\"] = pesos[\"hype_chat\"] * 1.0 * z(suavizar(s[\"hype\"], 8.0) / np.sqrt(base))\n        senales.append(chat.origen)\n\n    if cortes is not None and cortes.any():\n        comp[\"escenas\"] = pesos[\"escenas\"] * z(suavizar(cortes, 6.0))\n        senales.append(\"cortes de cámara\")\n\n    calor = _mapa_de_calor(info, n)\n    if calor is not None:\n        comp[\"calor\"] = 1.6 * z(calor)\n        senales.append(\"lo más visto de YouTube\")\n\n    puntos = sum(comp.values())\n    silencio = volumen < (np.median(volumen) - 20)\n    puntos = puntos - 1.5 * suavizar(silencio.astype(np.float32), 3.0)\n    return Analisis(puntos=suavizar(puntos, 1.5), volumen=volumen, componentes=comp, senales=senales)\n\n\n# ---------------------------------------------------------------------------\n# Elección de momentos\n# ---------------------------------------------------------------------------\n\n@dataclass\nclass Momento:\n    inicio: float\n    fin: float\n    pico: float\n    valor: float\n    motivo: str = \"\"\n    nota: int = 0\n    transcripcion: Optional[list] = None  # [(ini, fin, texto)] relativo al clip\n\n\ndef elegir_momentos(an: Analisis, cantidad: int, dur_min: float, dur_max: float,\n                    total: float) -> list[Momento]:\n    \"\"\"Busca picos de interés y construye un clip alrededor de cada uno.\n\n    El clip arranca un poco antes (para que se entienda el contexto) y acaba cuando\n    pasa la reacción, ajustando los cortes a pausas para no partir frases.\n    \"\"\"\n    s = an.puntos\n    n = len(s)\n    if total <= dur_min + 1 or n < 4:\n        return [Momento(0.0, total, 0.0, 1.0)]\n\n    mediana = float(np.median(s))\n    libre = np.ones(n, dtype=bool)\n    ocupado: list[tuple[float, float]] = []\n    momentos: list[Momento] = []\n    hueco = 2.0\n\n    for _ in range(cantidad * 6):\n        if len(momentos) >= cantidad:\n            break\n        candidatos = np.where(libre, s, -np.inf)\n        p = int(np.argmax(candidatos))\n        if not np.isfinite(candidatos[p]):\n            break\n        # Zona \"caliente\" alrededor del pico.\n        umbral = mediana + 0.5 * (s[p] - mediana)\n        izq = p\n        while izq > 0 and s[izq - 1] > umbral and (p - izq) * PASO < dur_max:\n            izq -= 1\n        der = p\n        while der < n - 1 and s[der + 1] > umbral and (der - p) * PASO < dur_max:\n            der += 1\n        a = izq * PASO - 6.0   # contexto previo\n        b = der * PASO + 3.0   # dejar ver la reacción\n        a, b = _ajustar_longitud(a, b, p * PASO, dur_min, dur_max, total)\n        a = _a_pausa(an.volumen, a, 1.5)\n        b = _a_pausa(an.volumen, b, 1.5)\n        if b - a < dur_min:\n            a, b = _ajustar_longitud(a, b, p * PASO, dur_min, dur_max, total)\n        a, b = _recortar_solapes(a, b, p * PASO, ocupado, hueco)\n        libre[max(0, int((p * PASO - 4) / PASO)):int((p * PASO + 4) / PASO) + 1] = False\n        if b - a < max(5.0, dur_min * 0.75):\n            continue\n        ocupado.append((a, b))\n        ia, ib = int(a / PASO), max(int(a / PASO) + 1, int(b / PASO))\n        libre[max(0, ia - int(hueco / PASO)):ib + int(hueco / PASO)] = False\n        valor = float(0.6 * s[p] + 0.4 * np.mean(s[ia:ib]))\n        momentos.append(Momento(round(a, 2), round(b, 2), p * PASO, valor))\n\n    calificar(momentos)\n    return momentos\n\n\ndef calificar(momentos: list[Momento]) -> None:\n    if not momentos:\n        return\n    vmin = min(m.valor for m in momentos)\n    vmax = max(m.valor for m in momentos)\n    for m in momentos:\n        m.nota = 100 if vmax == vmin else round(55 + 45 * (m.valor - vmin) / (vmax - vmin))\n\n\ndef _ajustar_longitud(a: float, b: float, pico: float, dmin: float, dmax: float,\n                      total: float) -> tuple[float, float]:\n    largo = b - a\n    if largo < dmin:\n        falta = dmin - largo\n        a -= falta * 0.65\n        b += falta * 0.35\n    elif largo > dmax:\n        a = max(a, pico - dmax * 0.65)\n        b = a + dmax\n    if a < 0:\n        b, a = b - a, 0.0\n    if b > total:\n        a, b = max(0.0, a - (b - total)), total\n    return a, b\n\n\ndef _a_pausa(volumen: np.ndarray, t: float, margen: float) -> float:\n    i = int(t / PASO)\n    k = int(margen / PASO)\n    lo, hi = max(0, i - k), min(len(volumen), i + k + 1)\n    if lo >= hi:\n        return t\n    return (lo + int(np.argmin(volumen[lo:hi]))) * PASO\n\n\ndef _recortar_solapes(a: float, b: float, pico: float, ocupado: list[tuple[float, float]],\n                      hueco: float) -> tuple[float, float]:\n    for oa, ob in ocupado:\n        if a < ob + hueco and b > oa - hueco:\n            if pico >= ob:\n                a = max(a, ob + hueco)\n            else:\n                b = min(b, oa - hueco)\n    return a, b\n\n\n# ---------------------------------------------------------------------------\n# Explicación\n# ---------------------------------------------------------------------------\n\nETIQUETAS = {\n    \"risa_chat\": \"😂 El chat se partió de risa\",\n    \"hype_chat\": \"🔥 Hype en el chat\",\n    \"chat\": \"💬 El chat explotó\",\n    \"voz\": \"📢 Gritos / subidón de voz\",\n    \"calor\": \"▶️ Lo más repetido del vídeo\",\n    \"escenas\": \"🎬 Mucha acción en pantalla\",\n}\n\n\ndef explicar(an: Analisis, m: Momento, chat: Optional[Chat]) -> str:\n    ia, ib = int(m.inicio / PASO), max(int(m.inicio / PASO) + 1, int(m.fin / PASO))\n    medias = {k: float(np.mean(np.sort(v[ia:ib])[-int(6 / PASO):])) for k, v in an.componentes.items()}\n    razones = [ETIQUETAS[k] for k, v in sorted(medias.items(), key=lambda kv: -kv[1])\n               if v > 0.8 and k in ETIQUETAS][:2]\n    if chat:\n        top = chat.palabras_destacadas(m.inicio, m.fin)\n        if top:\n            razones.append(\" \".join(f\"{p}×{c}\" for p, c in top))\n    return \" · \".join(razones) or \"Momento con más intensidad\"\n",
"clipper/chat.py": "\"\"\"Chat del stream: la mejor pista de qué momentos fueron graciosos o épicos.\n\n* Twitch VOD (directo resubido): se descarga la repetición del chat.\n* Twitch en directo: se escucha el chat (anónimo) mientras se graba.\n* YouTube (directos guardados): repetición del chat vía yt-dlp.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport glob\nimport json\nimport os\nimport re\nimport socket\nimport subprocess\nimport sys\nimport threading\nimport time\nimport urllib.request\nfrom collections import Counter\nfrom concurrent.futures import ThreadPoolExecutor\nfrom dataclasses import dataclass, field\nfrom typing import Optional\n\nMensaje = tuple[float, str]  # (segundo del vídeo, texto)\n\n# Emotes y expresiones típicas. Se comparan palabra a palabra en minúsculas.\nRISA = re.compile(\n    r\"^(?:lul|lulw|kekw|kek|kekl|omegalul|lmao+|lmfao|lol+|rofl|icant|pepelaugh|\"\n    r\"xd+|x+d+|kekwait|aware|clueless|\"\n    r\"(?:ja|je|ji|ha|he|js|aj|xa){2,}[jahes]*|jsjs\\w*|jajaj\\w*|\"\n    r\"😂+|🤣+|💀+|:face-with-tears-of-joy:|:rolling-on-the-floor-laughing:|:skull:)$\"\n)\nHYPE = re.compile(\n    r\"^(?:pog\\w*|poggers|pogu|pogchamp|w+|gg+|clip|clipped|clipalo|clipea|omg+|wtf+|\"\n    r\"no+o+|holy|hype|monkas|monkaw|letsgo|vamo+s*|\"\n    r\"🔥+|😱+|🤯+|😮+|💯+|:fire:|:exploding-head:|:face-screaming-in-fear:)$\"\n)\n\nRETRASO_CHAT = 5.0  # el chat reacciona unos segundos después de lo que pasa\n\n\n@dataclass\nclass Chat:\n    mensajes: list[Mensaje] = field(default_factory=list)\n    origen: str = \"\"\n\n    def __bool__(self) -> bool:\n        return len(self.mensajes) >= 30\n\n    def palabras_destacadas(self, a: float, b: float, n: int = 2) -> list[tuple[str, int]]:\n        \"\"\"Emotes de risa/hype más repetidos entre a y b (segundos del vídeo).\"\"\"\n        cuenta: Counter = Counter()\n        for t, texto in self.mensajes:\n            if a + RETRASO_CHAT * 0.4 <= t <= b + RETRASO_CHAT:\n                for p in set(_palabras(texto)):\n                    if RISA.match(p) or HYPE.match(p):\n                        cuenta[_normal(p)] += 1\n        return cuenta.most_common(n)\n\n\ndef _palabras(texto: str) -> list[str]:\n    return [p for p in re.split(r\"[\\s,.!¡?¿]+\", texto.lower()) if p]\n\n\ndef _normal(p: str) -> str:\n    if re.fullmatch(r\"(?:ja|je|ha|aj|js)+[jahes]*\", p):\n        return \"JAJAJA\"\n    if re.fullmatch(r\"x+d+\", p):\n        return \"XD\"\n    return p.upper() if p.isascii() else p\n\n\ndef clasificar(texto: str) -> tuple[bool, bool]:\n    \"\"\"(es_risa, es_hype) para un mensaje.\"\"\"\n    risa = hype = False\n    for p in _palabras(texto):\n        if RISA.match(p):\n            risa = True\n        elif HYPE.match(p):\n            hype = True\n    if not risa and re.search(r\"(?:ja){3,}|(?:je){3,}|(?:ha){3,}|😂|🤣|💀\", texto.lower()):\n        risa = True\n    return risa, hype\n\n\n# ---------------------------------------------------------------------------\n# Twitch VOD\n# ---------------------------------------------------------------------------\n\nGQL = \"https://gql.twitch.tv/gql\"\nCLIENT_IDS = [\"kd1unb4b3q4t58fwlpcbzcbnm76a8fp\", \"kimne78kx3ncx6brgo4mv6wki5h1ko\"]\nHASH_COMENTARIOS = \"b70a3591ff0f4e0313d126c6a1502d79a1c02baebb288227c582044aa76adf6a\"\n\n\ndef id_vod_twitch(info: dict) -> Optional[str]:\n    if \"twitch\" not in (info.get(\"extractor_key\") or info.get(\"extractor\") or \"\").lower():\n        return None\n    m = re.search(r\"(?:videos/|^v)(\\d+)\", str(info.get(\"id\", \"\")) + \" \" + (info.get(\"webpage_url\") or \"\"))\n    return m.group(1) if m else None\n\n\ndef _gql(cuerpo: list | dict, client_id: str) -> dict | list:\n    datos = json.dumps(cuerpo).encode()\n    req = urllib.request.Request(GQL, data=datos, method=\"POST\", headers={\n        \"Client-Id\": client_id, \"Content-Type\": \"application/json\",\n        \"User-Agent\": \"Mozilla/5.0\",\n    })\n    with urllib.request.urlopen(req, timeout=20) as r:\n        return json.loads(r.read().decode())\n\n\ndef _pagina_twitch(vod: str, client_id: str, offset: Optional[int] = None,\n                   cursor: Optional[str] = None) -> tuple[list[Mensaje], Optional[str]]:\n    variables: dict = {\"videoID\": vod}\n    if cursor:\n        variables[\"cursor\"] = cursor\n    else:\n        variables[\"contentOffsetSeconds\"] = offset or 0\n    respuesta = _gql([{\n        \"operationName\": \"VideoCommentsByOffsetOrCursor\",\n        \"variables\": variables,\n        \"extensions\": {\"persistedQuery\": {\"version\": 1, \"sha256Hash\": HASH_COMENTARIOS}},\n    }], client_id)\n    bloque = respuesta[0] if isinstance(respuesta, list) else respuesta\n    if bloque.get(\"errors\"):\n        raise RuntimeError(bloque[\"errors\"][0].get(\"message\", \"error GQL\"))\n    comentarios = (((bloque.get(\"data\") or {}).get(\"video\") or {}).get(\"comments\")) or {}\n    mensajes: list[Mensaje] = []\n    siguiente = None\n    for borde in comentarios.get(\"edges\") or []:\n        nodo = borde.get(\"node\") or {}\n        texto = \"\".join(f.get(\"text\", \"\") for f in ((nodo.get(\"message\") or {}).get(\"fragments\") or []))\n        mensajes.append((float(nodo.get(\"contentOffsetSeconds\", 0)), texto))\n        siguiente = borde.get(\"cursor\")\n    if not (comentarios.get(\"pageInfo\") or {}).get(\"hasNextPage\"):\n        siguiente = None\n    return mensajes, siguiente\n\n\ndef chat_twitch_vod(vod: str, duracion: float, limite_segundos: float = 240,\n                    progreso=None) -> Chat:\n    \"\"\"Descarga la repetición del chat en paralelo (trozos de 10 min).\"\"\"\n    client_id = None\n    for cid in CLIENT_IDS:\n        try:\n            _pagina_twitch(vod, cid, offset=0)\n            client_id = cid\n            break\n        except Exception:\n            continue\n    if client_id is None:\n        return Chat(origen=\"twitch (no disponible)\")\n\n    fin_limite = time.time() + limite_segundos\n    trozo = 600\n    tramos = [(a, min(duracion, a + trozo)) for a in range(0, int(duracion) + 1, trozo)]\n    hechos = [0]\n\n    def bajar(tramo: tuple[float, float]) -> list[Mensaje]:\n        a, b = tramo\n        salida: list[Mensaje] = []\n        cursor = None\n        primera = True\n        while time.time() < fin_limite:\n            try:\n                msgs, cursor = _pagina_twitch(vod, client_id, offset=int(a) if primera else None,\n                                              cursor=None if primera else cursor)\n            except Exception:\n                break\n            primera = False\n            salida += [m for m in msgs if a <= m[0] < b]\n            if not cursor or not msgs or msgs[-1][0] >= b:\n                break\n        hechos[0] += 1\n        if progreso:\n            progreso(hechos[0] * 100 / len(tramos), \"Leyendo el chat del directo…\")\n        return salida\n\n    with ThreadPoolExecutor(max_workers=8) as pool:\n        partes = list(pool.map(bajar, tramos))\n    mensajes = sorted(m for parte in partes for m in parte)\n    return Chat(mensajes=mensajes, origen=\"chat de Twitch\")\n\n\n# ---------------------------------------------------------------------------\n# Twitch en directo (IRC anónimo)\n# ---------------------------------------------------------------------------\n\ndef canal_twitch(url: str) -> Optional[str]:\n    m = re.search(r\"twitch\\.tv/([A-Za-z0-9_]{3,25})/?(?:$|\\?)\", url)\n    if m and m.group(1).lower() not in {\"videos\", \"directory\", \"settings\"}:\n        return m.group(1).lower()\n    return None\n\n\nclass OyenteTwitch(threading.Thread):\n    \"\"\"Guarda los mensajes del chat de un directo mientras se graba.\"\"\"\n\n    def __init__(self, canal: str):\n        super().__init__(daemon=True)\n        self.canal = canal\n        self.chat = Chat(origen=\"chat de Twitch (en directo)\")\n        self.inicio = time.time()\n        self._parar = threading.Event()\n\n    def parar(self) -> Chat:\n        self._parar.set()\n        self.join(timeout=5)\n        return self.chat\n\n    def run(self) -> None:\n        try:\n            s = socket.create_connection((\"irc.chat.twitch.tv\", 6667), timeout=10)\n            s.sendall(f\"NICK justinfan{int(time.time()) % 100000}\\r\\nJOIN #{self.canal}\\r\\n\".encode())\n            s.settimeout(1.0)\n            resto = \"\"\n            while not self._parar.is_set():\n                try:\n                    datos = s.recv(65536).decode(\"utf-8\", \"replace\")\n                except socket.timeout:\n                    continue\n                if not datos:\n                    break\n                resto += datos\n                *lineas, resto = resto.split(\"\\r\\n\")\n                for linea in lineas:\n                    if linea.startswith(\"PING\"):\n                        s.sendall(b\"PONG :tmi.twitch.tv\\r\\n\")\n                    elif \" PRIVMSG #\" in linea:\n                        texto = linea.split(\" :\", 1)[1] if \" :\" in linea else \"\"\n                        self.chat.mensajes.append((time.time() - self.inicio, texto))\n            s.close()\n        except OSError:\n            pass\n\n\n# ---------------------------------------------------------------------------\n# YouTube (repetición del chat de directos)\n# ---------------------------------------------------------------------------\n\ndef tiene_chat_youtube(info: dict) -> bool:\n    return \"live_chat\" in (info.get(\"subtitles\") or {})\n\n\ndef chat_youtube(url: str, carpeta: str, limite_segundos: float = 300) -> Chat:\n    os.makedirs(carpeta, exist_ok=True)\n    cmd = [sys.executable, \"-m\", \"yt_dlp\", \"--skip-download\", \"--write-subs\",\n           \"--sub-langs\", \"live_chat\", \"--no-playlist\", \"-q\", \"--no-warnings\",\n           \"-o\", os.path.join(carpeta, \"chat.%(ext)s\"), url]\n    try:\n        subprocess.run(cmd, timeout=limite_segundos, stdout=subprocess.DEVNULL,\n                       stderr=subprocess.DEVNULL)\n    except subprocess.TimeoutExpired:\n        pass  # usamos lo que haya dado tiempo a bajar\n    mensajes: list[Mensaje] = []\n    for ruta in glob.glob(os.path.join(carpeta, \"chat*live_chat.json*\")):\n        with open(ruta, encoding=\"utf-8\", errors=\"replace\") as f:\n            for linea in f:\n                m = _mensaje_youtube(linea)\n                if m:\n                    mensajes.append(m)\n    return Chat(mensajes=sorted(mensajes), origen=\"chat de YouTube\")\n\n\ndef _mensaje_youtube(linea: str) -> Optional[Mensaje]:\n    try:\n        d = json.loads(linea)\n    except ValueError:\n        return None\n    accion = d.get(\"replayChatItemAction\") or {}\n    try:\n        t = int(accion.get(\"videoOffsetTimeMsec\", -1)) / 1000\n    except (TypeError, ValueError):\n        return None\n    if t < 0:\n        return None\n    for a in accion.get(\"actions\") or []:\n        item = (a.get(\"addChatItemAction\") or {}).get(\"item\") or {}\n        render = item.get(\"liveChatTextMessageRenderer\") or item.get(\"liveChatPaidMessageRenderer\")\n        if not render:\n            continue\n        partes = []\n        for run in (render.get(\"message\") or {}).get(\"runs\") or []:\n            if \"text\" in run:\n                partes.append(run[\"text\"])\n            elif \"emoji\" in run:\n                e = run[\"emoji\"]\n                atajo = (e.get(\"shortcuts\") or [\"\"])[0]\n                partes.append(e.get(\"emojiId\") if not (e.get(\"emojiId\") or \"\").isascii() else atajo)\n        return t, \" \".join(partes)\n    return None\n",
"clipper/core.py": "\"\"\"Proceso completo de AutoClips: enlace/archivo -> momentos destacados -> clips listos.\n\nTodo gratis y local:\n  * yt-dlp  -> YouTube, Twitch (directos y VODs), Kick, TikTok, X, ...\n  * ffmpeg  -> audio, vídeo y cortes\n  * numpy   -> puntuación de cada momento\n  * faster-whisper (opcional) -> detecta risas en la voz y genera subtítulos\n\"\"\"\n\nfrom __future__ import annotations\n\nimport json\nimport os\nimport threading\nimport zipfile\nfrom dataclasses import asdict, dataclass, field\nfrom typing import Optional, Union\n\nfrom . import voz\nfrom .analisis import ESTILOS, Momento, calificar, cambios_de_escena, elegir_momentos, energia_audio, explicar, puntuar\nfrom .chat import Chat, OyenteTwitch, canal_twitch, chat_twitch_vod, chat_youtube, id_vod_twitch, tiene_chat_youtube\nfrom .cortes import FORMATOS, cortar_clip\nfrom .medios import (ClipError, Fuente, Progreso, descargar_audio, descargar_video, duracion_archivo,\n                     formato_video, fuente_local, fuente_remota, grabar_directo, obtener_info,\n                     sin_progreso, subprogreso)\n\nDURACION_AUTO = (15.0, 60.0)\n\n\n@dataclass\nclass Opciones:\n    cantidad: int = 5\n    duracion: Union[str, float] = \"auto\"  # \"auto\" (según el momento) o segundos fijos\n    formato: str = \"vertical_fondo\"\n    estilo: str = \"todo\"\n    ia_voz: bool = True        # detectar risas/reacciones con IA (si está instalada)\n    subtitulos: bool = False\n    escenas: bool = True       # cortes de cámara (solo con vídeo descargado/subido)\n    chat: bool = True\n    minutos_directo: float = 10\n    calidad_max: int = 1080\n    titulo: Optional[str] = None\n\n\n@dataclass\nclass Clip:\n    numero: int\n    archivo: str\n    miniatura: str\n    inicio: float\n    fin: float\n    puntuacion: int\n    motivo: str = \"\"\n    texto: Optional[str] = None\n\n\n@dataclass\nclass Resultado:\n    titulo: str\n    duracion_fuente: float\n    senales: list[str]\n    clips: list[Clip] = field(default_factory=list)\n    zip: Optional[str] = None\n    avisos: list[str] = field(default_factory=list)\n\n    def a_dict(self) -> dict:\n        return asdict(self)\n\n\ndef _rango_duracion(op: Opciones) -> tuple[float, float]:\n    if str(op.duracion).lower() == \"auto\":\n        return DURACION_AUTO\n    d = max(5.0, min(600.0, float(op.duracion)))\n    return d, d\n\n\ndef procesar(origen: str, carpeta: str, op: Optional[Opciones] = None,\n             progreso: Progreso = sin_progreso) -> Resultado:\n    \"\"\"`origen` es un enlace o la ruta de un vídeo. Deja los clips y un ZIP en `carpeta`.\"\"\"\n    op = op or Opciones()\n    if op.formato not in FORMATOS:\n        raise ClipError(f\"Formato desconocido: {op.formato}\")\n    if op.estilo not in ESTILOS:\n        op.estilo = \"todo\"\n    op.cantidad = max(1, min(50, int(op.cantidad)))\n    os.makedirs(carpeta, exist_ok=True)\n    trabajo = os.path.join(carpeta, \"fuente\")\n    avisos: list[str] = []\n\n    info: dict = {}\n    chat: Optional[Chat] = None\n    video_local: Optional[str] = None\n    fuente: Optional[Fuente] = None\n    audio: str\n\n    # 1) Conseguir el audio (para analizar), el chat y de dónde cortar el vídeo.\n    if os.path.isfile(origen):\n        video_local = origen\n        info = {\"title\": op.titulo or os.path.basename(origen)}\n        p_analisis = 0.0\n    else:\n        progreso(1, \"Leyendo el enlace…\")\n        info = obtener_info(origen, formato_video(op.calidad_max))\n        if info.get(\"is_live\"):\n            canal = canal_twitch(origen)\n            oyente = OyenteTwitch(canal) if (canal and op.chat) else None\n            if oyente:\n                oyente.start()\n            video_local = grabar_directo(info, trabajo, op.minutos_directo, subprogreso(progreso, 2, 40))\n            if oyente:\n                chat = oyente.parar()\n            p_analisis = 40.0\n        else:\n            chat_hilo: dict = {}\n            hilo = None\n            if op.chat:\n                hilo = threading.Thread(target=_leer_chat, args=(origen, info, trabajo, chat_hilo),\n                                        daemon=True)\n                hilo.start()\n            audio = descargar_audio(origen, trabajo, subprogreso(progreso, 2, 33))\n            if hilo:\n                progreso(33, \"Leyendo el chat del directo…\")\n                hilo.join()\n                chat = chat_hilo.get(\"chat\")\n            fuente = fuente_remota(info)\n            p_analisis = 35.0\n\n    if video_local:\n        fuente = fuente_local(video_local)\n        audio = video_local\n    if chat is not None and not chat:\n        chat = None\n\n    # 2) Analizar.\n    total = duracion_archivo(audio)\n    if total < 5:\n        raise ClipError(\"El vídeo es demasiado corto.\")\n    dur_min, dur_max = _rango_duracion(op)\n    if dur_min >= total:\n        dur_min = dur_max = total\n        avisos.append(\"El vídeo es más corto que la duración pedida: se genera un único clip.\")\n\n    usar_escenas = op.escenas and video_local is not None\n    fin_audio = p_analisis + (12 if usar_escenas else 15)\n    volumen = energia_audio(audio, total, subprogreso(progreso, p_analisis, fin_audio))\n    cortes = None\n    if usar_escenas:\n        cortes = cambios_de_escena(video_local, len(volumen), total,\n                                   subprogreso(progreso, fin_audio, fin_audio + 12))\n        fin_audio += 12\n    progreso(fin_audio, \"Buscando los mejores momentos…\")\n    an = puntuar(volumen, info, chat, cortes, op.estilo)\n\n    usar_voz = (op.ia_voz or op.subtitulos) and voz.disponible()\n    if (op.ia_voz or op.subtitulos) and not voz.disponible():\n        avisos.append(\"IA de voz no instalada: sin detección de risas ni subtítulos \"\n                      \"(instala 'faster-whisper', ver README).\")\n    extra = op.cantidad + 3 if (usar_voz and op.ia_voz) else 0\n    momentos = elegir_momentos(an, op.cantidad + extra, dur_min, dur_max, total)\n    if not momentos:\n        raise ClipError(\"No se encontraron momentos destacados.\")\n    if len(momentos) < op.cantidad:\n        avisos.append(f\"Solo se encontraron {len(momentos)} momentos que no se repitan.\")\n\n    # 3) IA de voz: re-ordenar los candidatos según risas/reacciones habladas.\n    motivos_voz: dict[int, str] = {}\n    p_voz = fin_audio + 2\n    if usar_voz and op.ia_voz:\n        p_fin_voz = p_voz + 20\n        for k, m in enumerate(momentos):\n            progreso(p_voz + (p_fin_voz - p_voz) * k / len(momentos),\n                     f\"Escuchando candidatos con IA ({k + 1}/{len(momentos)})…\")\n            try:\n                m.transcripcion = voz.transcribir(audio, m.inicio, m.fin)\n            except Exception as exc:  # sin internet la 1ª vez para bajar el modelo, etc.\n                avisos.append(f\"La IA de voz no pudo cargarse (la 1ª vez necesita internet) y se omitió: {exc}\")\n                usar_voz = False\n                break\n            bonus, motivo = voz.bonus_gracia(m.transcripcion)\n            peso = 1.6 if op.estilo == \"graciosos\" else 1.0\n            m.valor += bonus * peso\n            if motivo:\n                motivos_voz[id(m)] = motivo\n        p_voz = p_fin_voz\n    momentos = sorted(momentos, key=lambda m: -m.valor)[:op.cantidad]\n    calificar(momentos)\n\n    # 4) Cortar.\n    resultado = Resultado(titulo=info.get(\"title\") or os.path.basename(origen),\n                          duracion_fuente=total, senales=an.senales, avisos=avisos)\n    if usar_voz and op.ia_voz:\n        resultado.senales.append(\"risas en la voz (IA)\")\n    inicio_cortes = p_voz + 1\n    tramo = (97 - inicio_cortes) / len(momentos)\n    for k, m in enumerate(momentos, 1):\n        progreso(inicio_cortes + tramo * (k - 1), f\"Creando clip {k} de {len(momentos)}…\")\n        subs = None\n        if op.subtitulos and usar_voz:\n            try:\n                subs = m.transcripcion or voz.transcribir(audio, m.inicio, m.fin)\n            except Exception as exc:\n                avisos.append(f\"No se pudieron crear subtítulos ({exc}).\")\n        nombre = f\"clip_{k:02d}.mp4\"\n        salida = os.path.join(carpeta, nombre)\n        try:\n            cortar_clip(fuente, salida, m.inicio, m.fin, op.formato, subs)\n        except ClipError:\n            if fuente is None or fuente.local:\n                raise\n            # La web no deja cortar a distancia: descargamos el vídeo entero y seguimos.\n            progreso(inicio_cortes, \"Descargando el vídeo completo…\")\n            fuente = fuente_local(descargar_video(origen, trabajo, sin_progreso, op.calidad_max))\n            cortar_clip(fuente, salida, m.inicio, m.fin, op.formato, subs)\n\n        motivo = explicar(an, m, chat)\n        if id(m) in motivos_voz:\n            motivo = motivos_voz[id(m)] + \" · \" + motivo\n        resultado.clips.append(Clip(\n            numero=k, archivo=nombre, miniatura=f\"clip_{k:02d}.jpg\",\n            inicio=m.inicio, fin=m.fin, puntuacion=m.nota, motivo=motivo,\n            texto=\" \".join(t for _, _, t in m.transcripcion) if m.transcripcion else None,\n        ))\n\n    progreso(98, \"Empaquetando ZIP…\")\n    with zipfile.ZipFile(os.path.join(carpeta, \"clips.zip\"), \"w\", zipfile.ZIP_STORED) as z:\n        for c in resultado.clips:\n            z.write(os.path.join(carpeta, c.archivo), c.archivo)\n    resultado.zip = \"clips.zip\"\n    with open(os.path.join(carpeta, \"resultado.json\"), \"w\", encoding=\"utf-8\") as f:\n        json.dump(resultado.a_dict(), f, ensure_ascii=False, indent=2)\n    progreso(100, \"¡Clips listos!\")\n    return resultado\n\n\ndef _leer_chat(url: str, info: dict, carpeta: str, salida: dict) -> None:\n    try:\n        vod = id_vod_twitch(info)\n        if vod:\n            salida[\"chat\"] = chat_twitch_vod(vod, float(info.get(\"duration\") or 0) or 36000)\n        elif tiene_chat_youtube(info):\n            salida[\"chat\"] = chat_youtube(url, os.path.join(carpeta, \"chat\"))\n    except Exception:\n        pass  # sin chat también funciona, solo con el audio\n\n\n__all__ = [\"ClipError\", \"Clip\", \"ESTILOS\", \"FORMATOS\", \"Momento\", \"Opciones\", \"Resultado\", \"procesar\"]\n",
"clipper/cortes.py": "\"\"\"Corte de clips con ffmpeg, en horizontal o vertical, con subtítulos opcionales.\"\"\"\n\nfrom __future__ import annotations\n\nimport os\nfrom typing import Optional\n\nfrom .medios import Fuente, ffmpeg\n\nFORMATOS = {\n    \"vertical_fondo\": \"Vertical 9:16 con fondo desenfocado (TikTok / Reels / Shorts)\",\n    \"vertical\": \"Vertical 9:16 recortado al centro\",\n    \"original\": \"Original (horizontal)\",\n}\n\n\ndef _filtro(formato: str) -> str:\n    if formato == \"vertical\":\n        return \"crop='min(iw,ih*9/16)':'min(ih,iw*16/9)',scale=1080:1920,setsar=1\"\n    if formato == \"vertical_fondo\":\n        return (\"split[a][b];\"\n                \"[a]scale=1080:1920:force_original_aspect_ratio=increase,crop=1080:1920,\"\n                \"boxblur=20:2[fondo];\"\n                \"[b]scale=1080:1920:force_original_aspect_ratio=decrease[frente];\"\n                \"[fondo][frente]overlay=(W-w)/2:(H-h)/2,setsar=1\")\n    return \"scale=-2:'min(ih,1080)',setsar=1\"\n\n\ndef _tiempo_srt(t: float) -> str:\n    ms = int(round(max(0.0, t) * 1000))\n    h, ms = divmod(ms, 3_600_000)\n    m, ms = divmod(ms, 60_000)\n    s, ms = divmod(ms, 1000)\n    return f\"{h:02d}:{m:02d}:{s:02d},{ms:03d}\"\n\n\ndef escribir_srt(trozos: list, ruta: str) -> None:\n    with open(ruta, \"w\", encoding=\"utf-8\") as f:\n        for k, (a, b, texto) in enumerate(trozos, 1):\n            f.write(f\"{k}\\n{_tiempo_srt(a)} --> {_tiempo_srt(b)}\\n{texto.upper()}\\n\\n\")\n\n\ndef cortar_clip(fuente: Fuente, salida: str, inicio: float, fin: float,\n                formato: str = \"vertical_fondo\", subtitulos: Optional[list] = None) -> None:\n    \"\"\"Genera el clip MP4 (+ miniatura .jpg). `subtitulos`: [(ini, fin, texto)] del clip.\"\"\"\n    carpeta = os.path.dirname(os.path.abspath(salida))\n    base = os.path.splitext(os.path.basename(salida))[0]\n    duracion = max(0.5, fin - inicio)\n    filtro = _filtro(formato)\n\n    if subtitulos:\n        srt = f\"{base}.srt\"\n        escribir_srt(subtitulos, os.path.join(carpeta, srt))\n        tam = 14 if formato == \"original\" else 13\n        estilo = (f\"FontName=Arial,FontSize={tam},Bold=1,PrimaryColour=&H00FFFFFF,\"\n                  \"OutlineColour=&H00000000,BorderStyle=1,Outline=2,Shadow=0,\"\n                  \"Alignment=2,MarginV=60\")\n        filtro += f\",subtitles={srt}:force_style='{estilo}'\"\n\n    entradas, mapa_audio = fuente.args_entrada(inicio)\n    ffmpeg([\n        *entradas, \"-t\", f\"{duracion:.3f}\",\n        \"-filter_complex\", f\"[0:v]{filtro}[v]\", \"-map\", \"[v]\", *mapa_audio,\n        \"-c:v\", \"libx264\", \"-preset\", \"veryfast\", \"-crf\", \"21\", \"-pix_fmt\", \"yuv420p\",\n        \"-c:a\", \"aac\", \"-b:a\", \"160k\", \"-ar\", \"44100\", \"-movflags\", \"+faststart\",\n        os.path.basename(salida),\n    ], cwd=carpeta)\n\n    ffmpeg([\"-ss\", f\"{min(duracion / 2, 3):.3f}\", \"-i\", salida, \"-frames:v\", \"1\",\n            \"-vf\", \"scale=-2:360\", \"-q:v\", \"4\", os.path.join(carpeta, f\"{base}.jpg\")])\n",
"clipper/medios.py": "\"\"\"Acceso a vídeo: ffmpeg, yt-dlp, descargas, directos y fuentes remotas.\"\"\"\n\nfrom __future__ import annotations\n\nimport os\nimport re\nimport shutil\nimport subprocess\nfrom dataclasses import dataclass, field\nfrom typing import Callable, Optional\n\nProgreso = Callable[[float, str], None]\n\n\nclass ClipError(Exception):\n    \"\"\"Error con un mensaje pensado para mostrarse al usuario.\"\"\"\n\n\ndef sin_progreso(_pct: float, _msg: str) -> None:\n    pass\n\n\ndef subprogreso(progreso: Progreso, desde: float, hasta: float) -> Progreso:\n    \"\"\"Convierte el 0-100 de una fase en el tramo [desde, hasta] del total.\"\"\"\n    return lambda p, m: progreso(desde + (hasta - desde) * max(0.0, min(100.0, p)) / 100, m)\n\n\n# ---------------------------------------------------------------------------\n# ffmpeg\n# ---------------------------------------------------------------------------\n\ndef ffmpeg_bin() -> str:\n    \"\"\"Ruta de ffmpeg (el del sistema o el que trae imageio-ffmpeg).\"\"\"\n    ruta = shutil.which(\"ffmpeg\")\n    if ruta:\n        return ruta\n    try:\n        import imageio_ffmpeg\n\n        return imageio_ffmpeg.get_ffmpeg_exe()\n    except Exception as exc:  # pragma: no cover - depende del sistema\n        raise ClipError(\n            \"No se encontró ffmpeg. Instálalo (https://ffmpeg.org) o ejecuta \"\n            \"'pip install imageio-ffmpeg'.\"\n        ) from exc\n\n\ndef ffmpeg(args: list[str], cwd: Optional[str] = None) -> str:\n    cmd = [ffmpeg_bin(), \"-hide_banner\", \"-nostdin\", \"-y\", *args]\n    res = subprocess.run(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.PIPE)\n    if res.returncode != 0:\n        cola = res.stderr.decode(\"utf-8\", \"replace\")[-1500:]\n        raise ClipError(f\"ffmpeg falló:\\n{cola}\")\n    return res.stderr.decode(\"utf-8\", \"replace\")\n\n\ndef duracion_archivo(ruta: str) -> float:\n    res = subprocess.run(\n        [ffmpeg_bin(), \"-hide_banner\", \"-nostdin\", \"-i\", ruta],\n        stdout=subprocess.PIPE, stderr=subprocess.PIPE,\n    )\n    texto = res.stderr.decode(\"utf-8\", \"replace\")\n    m = re.search(r\"Duration:\\s*(\\d+):(\\d+):(\\d+(?:\\.\\d+)?)\", texto)\n    if not m:\n        raise ClipError(\"No se pudo leer la duración del vídeo (¿archivo dañado?).\")\n    h, mi, s = m.groups()\n    return int(h) * 3600 + int(mi) * 60 + float(s)\n\n\ndef _cabeceras(cab: dict) -> list[str]:\n    if not cab:\n        return []\n    return [\"-headers\", \"\".join(f\"{k}: {v}\\r\\n\" for k, v in cab.items())]\n\n\n# ---------------------------------------------------------------------------\n# Fuente de vídeo (archivo local o enlaces directos de la plataforma)\n# ---------------------------------------------------------------------------\n\n@dataclass\nclass Fuente:\n    \"\"\"De dónde sacar la imagen y el sonido de los clips.\n\n    `entradas` tiene una (vídeo+audio juntos) o dos (vídeo, audio) rutas/URLs.\n    Con URLs remotas, ffmpeg descarga solo el trozo que necesita cada clip.\n    \"\"\"\n\n    entradas: list[tuple[str, dict]]\n    local: bool = True\n    cabeceras: dict = field(default_factory=dict)\n\n    def args_entrada(self, inicio: float) -> tuple[list[str], list[str]]:\n        args: list[str] = []\n        for url, cab in self.entradas:\n            if not self.local:\n                args += _cabeceras(cab)\n            args += [\"-ss\", f\"{inicio:.3f}\", \"-i\", url]\n        audio = \"1:a:0\" if len(self.entradas) > 1 else \"0:a:0?\"\n        return args, [\"-map\", audio]\n\n\ndef fuente_local(ruta: str) -> Fuente:\n    return Fuente(entradas=[(os.path.abspath(ruta), {})], local=True)\n\n\n# ---------------------------------------------------------------------------\n# yt-dlp\n# ---------------------------------------------------------------------------\n\ndef _ydl_base() -> dict:\n    return {\n        \"quiet\": True,\n        \"no_warnings\": True,\n        \"noprogress\": True,\n        \"noplaylist\": True,\n        \"ffmpeg_location\": ffmpeg_bin(),\n        \"retries\": 5,\n        \"fragment_retries\": 10,\n    }\n\n\ndef formato_video(calidad_max: int) -> str:\n    return (f\"bv*[height<={calidad_max}]+ba/b[height<={calidad_max}]/\"\n            f\"bv*+ba/b\")\n\n\ndef obtener_info(url: str, formato: Optional[str] = None) -> dict:\n    import yt_dlp\n\n    opciones = _ydl_base()\n    if formato:\n        opciones[\"format\"] = formato\n    try:\n        with yt_dlp.YoutubeDL(opciones) as ydl:\n            info = ydl.extract_info(url, download=False)\n    except Exception as exc:\n        raise ClipError(\n            \"No se pudo leer el enlace. Comprueba que es público y correcto \"\n            f\"(los VODs solo para suscriptores no se pueden).\\n({exc})\"\n        ) from exc\n    if info.get(\"_type\") == \"playlist\":\n        entradas = [e for e in info.get(\"entries\") or [] if e]\n        if not entradas:\n            raise ClipError(\"La lista de reproducción está vacía.\")\n        return obtener_info(entradas[0].get(\"webpage_url\") or entradas[0][\"url\"], formato)\n    return info\n\n\ndef fuente_remota(info: dict) -> Optional[Fuente]:\n    \"\"\"Enlaces directos al vídeo elegido por yt-dlp, para cortar sin descargarlo entero.\"\"\"\n    formatos = info.get(\"requested_formats\") or ([info] if info.get(\"url\") else [])\n    if not formatos:\n        return None\n    if any(f.get(\"protocol\", \"\").startswith((\"http_dash_segments\", \"f4m\", \"ism\")) for f in formatos):\n        return None  # fragmentados raros: mejor descargar entero\n    entradas = [(f[\"url\"], f.get(\"http_headers\") or info.get(\"http_headers\") or {}) for f in formatos]\n    return Fuente(entradas=entradas, local=False)\n\n\ndef _gancho(progreso: Progreso, texto: str):\n    def gancho(d: dict) -> None:\n        if d.get(\"status\") == \"downloading\":\n            total = d.get(\"total_bytes\") or d.get(\"total_bytes_estimate\") or 0\n            hecho = d.get(\"downloaded_bytes\") or 0\n            if total:\n                progreso(min(99.0, hecho * 100 / total), texto)\n            elif d.get(\"fragment_count\"):\n                progreso(min(99.0, (d.get(\"fragment_index\") or 0) * 100 / d[\"fragment_count\"]), texto)\n    return gancho\n\n\ndef _buscar(carpeta: str, prefijo: str) -> str:\n    for nombre in sorted(os.listdir(carpeta)):\n        if nombre.startswith(prefijo + \".\") and not nombre.endswith((\".part\", \".ytdl\")):\n            return os.path.join(carpeta, nombre)\n    raise ClipError(\"La descarga terminó pero no se encontró el archivo.\")\n\n\ndef descargar_audio(url: str, carpeta: str, progreso: Progreso = sin_progreso) -> str:\n    \"\"\"Descarga solo el audio (rápido incluso en directos resubidos de horas).\"\"\"\n    import yt_dlp\n\n    os.makedirs(carpeta, exist_ok=True)\n    opciones = {\n        **_ydl_base(),\n        \"outtmpl\": os.path.join(carpeta, \"audio.%(ext)s\"),\n        \"format\": \"ba/ba*/wa*/w\",\n        \"concurrent_fragment_downloads\": 8,\n        \"progress_hooks\": [_gancho(progreso, \"Descargando el audio para analizarlo…\")],\n    }\n    try:\n        with yt_dlp.YoutubeDL(opciones) as ydl:\n            ydl.download([url])\n    except Exception as exc:\n        raise ClipError(f\"Falló la descarga del audio.\\n({exc})\") from exc\n    return _buscar(carpeta, \"audio\")\n\n\ndef descargar_video(url: str, carpeta: str, progreso: Progreso = sin_progreso,\n                    calidad_max: int = 1080) -> str:\n    import yt_dlp\n\n    os.makedirs(carpeta, exist_ok=True)\n    opciones = {\n        **_ydl_base(),\n        \"outtmpl\": os.path.join(carpeta, \"video.%(ext)s\"),\n        \"format\": formato_video(calidad_max),\n        \"merge_output_format\": \"mp4\",\n        \"concurrent_fragment_downloads\": 8,\n        \"progress_hooks\": [_gancho(progreso, \"Descargando vídeo…\")],\n    }\n    try:\n        with yt_dlp.YoutubeDL(opciones) as ydl:\n            ydl.download([url])\n    except Exception as exc:\n        raise ClipError(f\"Falló la descarga del vídeo.\\n({exc})\") from exc\n    return _buscar(carpeta, \"video\")\n\n\ndef grabar_directo(info: dict, carpeta: str, minutos: float,\n                   progreso: Progreso = sin_progreso) -> str:\n    \"\"\"Graba los próximos `minutos` de un directo con ffmpeg.\"\"\"\n    formatos = info.get(\"requested_formats\") or [info]\n    entradas: list[str] = []\n    for f in formatos:\n        entradas += _cabeceras(f.get(\"http_headers\") or info.get(\"http_headers\") or {})\n        entradas += [\"-i\", f[\"url\"]]\n\n    segundos = max(60, int(minutos * 60))\n    os.makedirs(carpeta, exist_ok=True)\n    salida = os.path.join(carpeta, \"directo.mp4\")\n    mapas = ([\"-map\", \"0:v:0?\", \"-map\", \"0:a:0?\"] if len(formatos) == 1\n             else [\"-map\", \"0:v:0\", \"-map\", \"1:a:0\"])\n    cmd = [ffmpeg_bin(), \"-hide_banner\", \"-nostdin\", \"-y\", *entradas, *mapas,\n           \"-t\", str(segundos), \"-c\", \"copy\", \"-progress\", \"pipe:1\", \"-nostats\", salida]\n    registro_ruta = os.path.join(carpeta, \"grabacion.log\")\n    with open(registro_ruta, \"w\", encoding=\"utf-8\") as registro:\n        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=registro, text=True)\n        assert proc.stdout is not None\n        for linea in proc.stdout:\n            if linea.startswith(\"out_time_us=\"):\n                try:\n                    seg = int(linea.split(\"=\", 1)[1]) / 1e6\n                except ValueError:\n                    continue\n                resta = max(0, segundos - seg)\n                progreso(min(99.0, seg * 100 / segundos),\n                         f\"Grabando directo… faltan {int(resta // 60)}:{int(resta % 60):02d}\")\n        proc.wait()\n    if not os.path.exists(salida) or os.path.getsize(salida) < 10_000:\n        with open(registro_ruta, encoding=\"utf-8\", errors=\"replace\") as f:\n            error = f.read()[-800:]\n        raise ClipError(f\"No se pudo grabar el directo.\\n{error}\")\n    return salida\n",
"clipper/voz.py": "\"\"\"IA de voz gratuita (faster-whisper): detecta risas/frases y crea subtítulos.\"\"\"\n\nfrom __future__ import annotations\n\nimport os\nimport re\nimport tempfile\nfrom typing import Optional\n\nfrom .medios import ffmpeg\n\nTrozo = tuple[float, float, str]\n\nMODELO = os.environ.get(\"AUTOCLIPS_MODELO\", \"base\")\n_modelo = None\n\n\ndef disponible() -> bool:\n    try:\n        import faster_whisper  # noqa: F401\n\n        return True\n    except Exception:\n        return False\n\n\ndef _cargar():\n    global _modelo\n    if _modelo is None:\n        from faster_whisper import WhisperModel\n\n        try:  # con tarjeta gráfica (p. ej. Google Colab) va mucho más rápido\n            import ctranslate2\n\n            if ctranslate2.get_cuda_device_count() > 0:\n                _modelo = WhisperModel(os.environ.get(\"AUTOCLIPS_MODELO\", \"small\"),\n                                       device=\"cuda\", compute_type=\"float16\")\n        except Exception:\n            _modelo = None\n        if _modelo is None:\n            _modelo = WhisperModel(MODELO, device=\"cpu\", compute_type=\"int8\")\n    return _modelo\n\n\ndef transcribir(audio: str, inicio: float, fin: float) -> list[Trozo]:\n    \"\"\"Transcribe [inicio, fin] del audio y lo agrupa en frases cortas estilo TikTok.\n\n    Los tiempos devueltos son relativos al inicio del clip.\n    \"\"\"\n    with tempfile.TemporaryDirectory() as tmp:\n        wav = os.path.join(tmp, \"trozo.wav\")\n        ffmpeg([\"-ss\", f\"{inicio:.3f}\", \"-i\", audio, \"-t\", f\"{fin - inicio:.3f}\",\n                \"-vn\", \"-ac\", \"1\", \"-ar\", \"16000\", wav])\n        segmentos, _ = _cargar().transcribe(wav, word_timestamps=True, vad_filter=False)\n        trozos: list[Trozo] = []\n        grupo: list = []\n        for seg in segmentos:\n            for p in seg.words or []:\n                grupo.append(p)\n                texto = \"\".join(x.word for x in grupo).strip()\n                if len(grupo) >= 4 or len(texto) > 22 or p.word.strip().endswith((\".\", \"?\", \"!\", \",\")):\n                    trozos.append((grupo[0].start, grupo[-1].end, texto))\n                    grupo = []\n        if grupo:\n            trozos.append((grupo[0].start, grupo[-1].end, \"\".join(x.word for x in grupo).strip()))\n    return trozos\n\n\nRISAS = re.compile(r\"(?:ja){2,}|(?:je){2,}|(?:ha){2,}|(?:jsjs)|\\b(?:risas?|laugh\\w*|lol|xd)\\b\", re.I)\nEXPRESIONES = re.compile(\n    r\"\\b(?:no puede ser|qu[eé] (?:dices|haces|ha pasado)|madre m[ií]a|dios m[ií]o|\"\n    r\"joder|hostia|la madre|wtf|oh my god|nooo+)\\b\", re.I)\n\n\ndef bonus_gracia(trozos: list[Trozo]) -> tuple[float, Optional[str]]:\n    \"\"\"Puntos extra si en el clip hay risas o reacciones habladas.\"\"\"\n    texto = \" \".join(t for _, _, t in trozos)\n    risas = len(RISAS.findall(texto))\n    exclamaciones = texto.count(\"!\") + texto.count(\"¡\")\n    expresiones = len(EXPRESIONES.findall(texto))\n    bonus = 0.7 * min(risas, 4) + 0.12 * min(exclamaciones, 6) + 0.35 * min(expresiones, 3)\n    if not texto.strip():\n        bonus -= 0.3\n    motivo = \"🤣 Risas en el audio\" if risas else (\"🗣️ Reacción hablada\" if expresiones else None)\n    return bonus, motivo\n"
}
DESTINO = "/content/AutoClips"
for ruta, contenido in CODIGO.items():
    completa = os.path.join(DESTINO, ruta)
    os.makedirs(os.path.dirname(completa), exist_ok=True)
    with open(completa, "w", encoding="utf-8") as f:
        f.write(contenido)
print("Instalando herramientas gratuitas (1-2 minutos)…")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "yt-dlp[default]",
                "flask", "faster-whisper"], check=False)
if subprocess.run(["which", "ffmpeg"], capture_output=True).returncode != 0:
    subprocess.run("apt-get -qq install -y ffmpeg > /dev/null", shell=True)
os.chdir(DESTINO)
if DESTINO not in sys.path:
    sys.path.insert(0, DESTINO)
print("✅ Listo. Ahora pulsa ▶ en el paso 2.")

In [ ]:
#@title 2️⃣ Abrir AutoClips (pulsa ▶ y toca el botón)
import os, threading, time
from IPython.display import HTML, display
import app as autoclips

os.makedirs(autoclips.TRABAJOS, exist_ok=True)
if not globals().get("_servidor_autoclips"):
    _servidor_autoclips = threading.Thread(
        target=lambda: autoclips.app.run(host="127.0.0.1", port=5000, threaded=True), daemon=True)
    _servidor_autoclips.start()
    time.sleep(2)

from google.colab.output import eval_js
enlace = eval_js("google.colab.kernel.proxyPort(5000)")
display(HTML(f"""
<a href="{enlace}" target="_blank" style="display:inline-block;padding:16px 28px;border-radius:12px;
   background:linear-gradient(90deg,#ff4d6d,#ff8a3d);color:#fff;font:700 20px system-ui;text-decoration:none">
   🎬 Abrir AutoClips</a>
<p style="font:14px system-ui">Se abre en otra pestaña. No cierres esta.</p>
"""))

---
¿Prefieres no abrir otra pestaña? Rellena esto y pulsa ▶:

In [ ]:
#@title 🅱️ (Alternativa) Sacar clips desde aquí, sin abrir otra pestaña
enlace = "" #@param {type:"string"}
cantidad = 5 #@param {type:"slider", min:1, max:30, step:1}
duracion = "auto" #@param ["auto", "15", "20", "30", "45", "60", "90"]
estilo = "todo" #@param ["todo", "graciosos", "epicos"]
formato = "vertical_fondo" #@param ["vertical_fondo", "vertical", "original"]
subtitulos = False #@param {type:"boolean"}
detectar_risas_con_ia = True #@param {type:"boolean"}
minutos_si_es_directo = 10 #@param {type:"integer"}
guardar_en_google_drive = False #@param {type:"boolean"}

import os, shutil, sys, time
from IPython.display import HTML, display
from clipper import ClipError, Opciones, procesar

if not enlace.strip():
    raise SystemExit("Pega un enlace en el campo 'enlace' y vuelve a pulsar ▶")
carpeta = f"/content/clips/{time.strftime('%Y%m%d_%H%M%S')}"

def progreso(p, m):
    sys.stdout.write(f"\r{p:5.1f}%  {m[:60]:<60}")
    sys.stdout.flush()

try:
    r = procesar(enlace.strip(), carpeta, Opciones(
        cantidad=cantidad, duracion=duracion, estilo=estilo, formato=formato,
        subtitulos=subtitulos, ia_voz=detectar_risas_con_ia,
        minutos_directo=minutos_si_es_directo), progreso)
except ClipError as e:
    raise SystemExit(f"\n❌ {e}")

print(f"\n\n✅ {len(r.clips)} clips de «{r.titulo}» · analizado con: {', '.join(r.senales)}")
for a in r.avisos:
    print("⚠️", a)
for c in r.clips:
    print(f"#{c.numero}  {c.fin - c.inicio:.0f}s  🔥{c.puntuacion}%  {c.motivo}")
if guardar_en_google_drive:
    from google.colab import drive
    drive.mount("/content/drive")
    destino = "/content/drive/MyDrive/AutoClips/" + os.path.basename(carpeta)
    shutil.copytree(carpeta, destino, ignore=shutil.ignore_patterns("fuente"))
    print("📁 Guardados en Google Drive:", destino)
from google.colab import files
files.download(os.path.join(carpeta, "clips.zip"))